# Titanic生存予測: EDAと特徴量エンジニアリング

このNotebookでは、Titanic乗客データを使って生存有無 `Survived` を予測します。目的は、EDAから仮説を立て、特徴量を作成し、複数モデルを比較する一連のデータ分析プロセスを確認することです。

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
DOWNLOADS_DIR = Path.home() / "Downloads"

## 1. データ読み込み

公開リポジトリには元CSVを含めていないため、`data/raw/` または `Downloads` に配置されたファイルを読み込みます。

In [ ]:
def resolve_data_path(filename):
    project_path = DATA_DIR / filename
    if project_path.exists():
        return project_path

    downloads_path = DOWNLOADS_DIR / filename
    if downloads_path.exists():
        return downloads_path

    raise FileNotFoundError(f"{filename} が見つかりません。{DATA_DIR} または {DOWNLOADS_DIR} に配置してください。")

train = pd.read_csv(resolve_data_path("train_local (1).csv"))
valid = pd.read_csv(resolve_data_path("valid (1).csv"))
eval_df = pd.read_csv(resolve_data_path("eval (1).csv"))
sample_submission = pd.read_csv(resolve_data_path("sample_submission.csv"))

for name, df in [("学習データ", train), ("検証データ", valid), ("予測対象データ", eval_df), ("提出サンプル", sample_submission)]:
    print(name, df.shape)
    print(df.isnull().sum())
    print()

## 2. EDAで確認する観点

`Sex`、`Pclass`、`Embarked`、`Cabin`、`Name` に含まれる敬称などが、生存率に関係している可能性があります。

In [ ]:
print("目的変数の分布")
print(train["Survived"].value_counts())
print(train["Survived"].value_counts(normalize=True).round(3))

print("\n性別ごとの生存率")
print(train.groupby("Sex")["Survived"].agg(["count", "mean"]).round(3))

print("\n客室等級ごとの生存率")
print(train.groupby("Pclass")["Survived"].agg(["count", "mean"]).round(3))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
train.groupby("Sex")["Survived"].mean().plot(kind="bar", ax=axes[0], title="性別ごとの生存率")
train.groupby("Pclass")["Survived"].mean().plot(kind="bar", ax=axes[1], title="客室等級ごとの生存率")
train.groupby("Embarked")["Survived"].mean().plot(kind="bar", ax=axes[2], title="乗船港ごとの生存率")
plt.tight_layout()

## 3. 特徴量エンジニアリング

EDAから得た仮説をもとに、家族人数、単独乗船、敬称、Cabin情報の有無、Deckを作成します。

In [ ]:
def extract_title(name):
    title = name.split(",")[1].split(".")[0].strip()
    rare_titles = {"Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major", "Rev", "Sir", "Jonkheer", "Dona"}
    if title in rare_titles:
        return "Rare"
    if title in {"Mlle", "Ms"}:
        return "Miss"
    if title == "Mme":
        return "Mrs"
    return title

def add_features(df):
    df = df.copy()
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
    df["Title"] = df["Name"].apply(extract_title)
    df["HasCabin"] = df["Cabin"].notna().astype(int)
    df["Deck"] = df["Cabin"].fillna("Unknown").astype(str).str[0]
    df.loc[df["Deck"] == "U", "Deck"] = "Unknown"
    return df

train_fe = add_features(train)
valid_fe = add_features(valid)
eval_fe = add_features(eval_df)
train_fe[["FamilySize", "IsAlone", "Title", "HasCabin", "Deck"]].head()

## 4. モデル比較

数値列は中央値補完と標準化、カテゴリ列は最頻値補完とOne-Hot Encodingを行い、ベースラインと特徴量追加後のモデルを比較します。

In [ ]:
def make_model(numeric_features, categorical_features, estimator):
    numeric_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
    categorical_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))])
    preprocessor = ColumnTransformer(transformers=[("num", numeric_transformer, numeric_features), ("cat", categorical_transformer, categorical_features)])
    return Pipeline(steps=[("preprocess", preprocessor), ("model", estimator)])

y_train = train["Survived"]
y_valid = valid["Survived"]
feature_numeric = ["Pclass", "Age", "Fare", "SibSp", "Parch", "FamilySize", "IsAlone", "HasCabin"]
feature_categorical = ["Sex", "Embarked", "Title", "Deck"]

experiments = [
    ("baseline_logreg", train, valid, ["Pclass", "Age", "Fare"], ["Sex", "Embarked"], LogisticRegression(max_iter=1000, random_state=42)),
    ("baseline_rf", train, valid, ["Pclass", "Age", "Fare"], ["Sex", "Embarked"], RandomForestClassifier(n_estimators=300, max_depth=5, min_samples_leaf=4, random_state=42)),
    ("feature_logreg", train_fe, valid_fe, feature_numeric, feature_categorical, LogisticRegression(max_iter=1000, random_state=42)),
    ("feature_rf", train_fe, valid_fe, feature_numeric, feature_categorical, RandomForestClassifier(n_estimators=500, max_depth=6, min_samples_leaf=3, random_state=42)),
    ("feature_gb", train_fe, valid_fe, feature_numeric, feature_categorical, GradientBoostingClassifier(n_estimators=120, learning_rate=0.04, max_depth=3, random_state=42)),
]

rows = []
fitted_models = {}
for name, train_data, valid_data, numeric, categorical, estimator in experiments:
    model = make_model(numeric, categorical, estimator)
    model.fit(train_data, y_train)
    pred = model.predict(valid_data)
    acc = accuracy_score(y_valid, pred)
    rows.append({"実験名": name, "Accuracy": acc, "数値特徴量": ", ".join(numeric), "カテゴリ特徴量": ", ".join(categorical)})
    fitted_models[name] = model

results = pd.DataFrame(rows).sort_values("Accuracy", ascending=False)
results

## 5. 予測CSVの作成と考察

最良モデルを使って `eval` データを予測し、提出用CSVを作成します。特徴量エンジニアリング後のモデルがベースラインより改善するかを確認します。

In [ ]:
best_name = results.iloc[0]["実験名"]
best_model = fitted_models[best_name]
best_eval_data = eval_fe if best_name.startswith("feature") else eval_df

eval_pred = best_model.predict(best_eval_data)
submission = pd.DataFrame({"PassengerId": eval_df["PassengerId"], "Survived": eval_pred.astype(int)})

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
results.to_csv(OUTPUT_DIR / "titanic_experiment_results.csv", index=False)
submission.to_csv(OUTPUT_DIR / "submission_eval.csv", index=False)

print("最良モデル:", best_name)
print("提出行数:", len(submission))
submission.head()